In [11]:
"""
The idea of this activity is to write an algorithm in a "distributed way", ie. operating on
a distributed tree. 

We highlight the differences with the sequential (or partitioned) version
and compare the execution times.

This is typically what you would do if you develop new functionnalities in maia.

"""

'\nThe idea of this activity is to write an algorithm in a "distributed way", ie. operating on\na distributed tree. \n\nWe highlight the differences with the sequential (or partitioned) version\nand compare the execution times.\n\nThis is typically what you would do if you develop new functionnalities in maia.\n\n'

In [12]:
import time
import numpy as np
from mpi4py import MPI
comm = MPI.COMM_WORLD

In [13]:
import maia
import maia.pytree as PT

In [14]:
class DIndexer:
    def __init__(self, distri, indices, comm):
        from maia.transfer import protocols as EP
        self.btp = EP.BlockToPart(distri, [indices], comm)

    def take(self, data_in):
        _, data_out = self.btp.exchange_field(data_in)
        return data_out[0]

In [15]:
def compute_cc_seq(tree):
    zone = PT.get_node_from_label(tree, 'Zone_t')

    cx, cy, cz = PT.Zone.coordinates(zone)
    connec = PT.get_node_from_name(zone, 'ElementConnectivity')[1]

    n_elem = connec.size // 4
    connec_idx = 4*np.arange(n_elem+1)

    mean_x = np.add.reduceat(np.take(cx, connec-1), connec_idx[:-1]) / 4
    mean_y = np.add.reduceat(np.take(cy, connec-1), connec_idx[:-1]) / 4
    mean_z = np.add.reduceat(np.take(cz, connec-1), connec_idx[:-1]) / 4

    PT.new_FlowSolution('Centers',
                        loc='CellCenter',
                        fields={'CCX' : mean_x, 'CCY' : mean_y, 'CCZ' : mean_z},
                        parent=zone)

In [16]:
def compute_cc_dist(dist_tree, comm):
    zone = PT.get_node_from_label(dist_tree, 'Zone_t')

    cx, cy, cz = PT.Zone.coordinates(zone)
    connec = PT.get_node_from_name(zone, 'ElementConnectivity')[1]

    vtx_distri = PT.maia.getDistribution(zone, 'Vertex')[1]
    indexer = DIndexer(vtx_distri, connec, comm)
    

    dn_elem = connec.size // 4
    connec_idx = 4*np.arange(dn_elem+1)

    mean_x = np.add.reduceat(indexer.take(cx), connec_idx[:-1]) / 4
    mean_y = np.add.reduceat(indexer.take(cy), connec_idx[:-1]) / 4
    mean_z = np.add.reduceat(indexer.take(cz), connec_idx[:-1]) / 4
    
    PT.new_FlowSolution('Centers',
                        loc='CellCenter',
                        fields={'CCX' : mean_x, 'CCY' : mean_y, 'CCZ' : mean_z},
                        parent=zone)
# NB : you can try with bigger meshes, first you need to generate it using
#dist_tree = maia.factory.generate_dist_block(101, 'TETRA_4', comm)
#maia.io.dist_tree_to_file(dist_tree, 'tetra100.hdf', comm)

In [17]:
FILENAME = 'tetra10.hdf'

In [18]:
# Sequential
if comm.rank == 0:
    tree = maia.io.read_tree('tetra10.hdf')
    compute_cc_seq(tree)
    maia.io.write_tree(tree, 'sol.hdf')
    PT.print_tree(tree)

CGNSTree CGNSTree_t 
├───CGNSLibraryVersion CGNSLibraryVersion_t R4 [4.2]
└───Base CGNSBase_t I4 [3 3]
    └───zone Zone_t I4 [[1331 5000    0]]
        ├───ZoneType ZoneType_t "Unstructured"
        ├───GridCoordinates GridCoordinates_t 
        │   ├───CoordinateX DataArray_t R8 (1331,)
        │   ├───CoordinateY DataArray_t R8 (1331,)
        │   └───CoordinateZ DataArray_t R8 (1331,)
        ├───TETRA_4.0 Elements_t I4 [10  0]
        │   ├───ElementRange IndexRange_t I4 [   1 5000]
        │   └───ElementConnectivity DataArray_t I4 (20000,)
        ├───TRI_3.0 Elements_t I4 [5 0]
        │   ├───ElementRange IndexRange_t I4 [5001 6200]
        │   └───ElementConnectivity DataArray_t I4 (3600,)
        ├───ZoneBC ZoneBC_t 
        │   ├───Zmin BC_t "Null"
        │   │   ├───GridLocation GridLocation_t "FaceCenter"
        │   │   └───PointList IndexArray_t I4 (1, 200)
        │   ├───Zmax BC_t "Null"
        │   │   ├───GridLocation GridLocation_t "FaceCenter"
        │   │   └──

In [19]:
# Parallel partitioned
tree = maia.io.file_to_dist_tree('tetra10.hdf', comm)
ptree = maia.factory.partition_dist_tree(tree, comm)
compute_cc_seq(ptree)
maia.transfer.part_tree_to_dist_tree_all(tree, ptree, comm)
maia.io.dist_tree_to_file(tree, 'sol.hdf', comm)
PT.print_tree(tree)

Distributed read of file tetra10.hdf...
Read completed (0.02 s) -- Size of dist_tree for current rank is 143.7KiB (Σ=143.7KiB)
Partitioning tree of 1 initial block...
Partitioning completed (0.05 s) -- Nb of cells for current rank is 5.0K (Σ=5.0K)
Distributed write of a 262.2KiB dist_tree (Σ=262.2KiB)...
CGNSTree CGNSTree_t 
Write completed [sol.hdf] (0.51 s)
├───CGNSLibraryVersion CGNSLibraryVersion_t R4 [4.2]
└───Base CGNSBase_t I4 [3 3]
    └───zone Zone_t I4 [[1331 5000    0]]
        ├───ZoneType ZoneType_t "Unstructured"
        ├───GridCoordinates GridCoordinates_t 
        │   ├───CoordinateX DataArray_t R8 (1331,)
        │   ├───CoordinateY DataArray_t R8 (1331,)
        │   └───CoordinateZ DataArray_t R8 (1331,)
        ├───TETRA_4.0 Elements_t I4 [10  0]
        │   ├───ElementRange IndexRange_t I4 [   1 5000]
        │   ├───ElementConnectivity DataArray_t I4 (20000,)
        │   └───:CGNS#Distribution UserDefinedData_t 
        │       └───Element DataArray_t I4 [   0 500

In [20]:
# Parallel distributed
tree = maia.io.file_to_dist_tree('tetra10.hdf', comm)
compute_cc_dist(tree, comm)
maia.io.dist_tree_to_file(tree, 'sol.hdf', comm)
PT.print_tree(tree)

Distributed read of file tetra10.hdf...
Read completed (0.02 s) -- Size of dist_tree for current rank is 143.7KiB (Σ=143.7KiB)
Distributed write of a 262.2KiB dist_tree (Σ=262.2KiB)...
CGNSTree CGNSTree_t 
Write completed [sol.hdf] (0.52 s)
├───CGNSLibraryVersion CGNSLibraryVersion_t R4 [4.2]
└───Base CGNSBase_t I4 [3 3]
    └───zone Zone_t I4 [[1331 5000    0]]
        ├───ZoneType ZoneType_t "Unstructured"
        ├───GridCoordinates GridCoordinates_t 
        │   ├───CoordinateX DataArray_t R8 (1331,)
        │   ├───CoordinateY DataArray_t R8 (1331,)
        │   └───CoordinateZ DataArray_t R8 (1331,)
        ├───TETRA_4.0 Elements_t I4 [10  0]
        │   ├───ElementRange IndexRange_t I4 [   1 5000]
        │   ├───ElementConnectivity DataArray_t I4 (20000,)
        │   └───:CGNS#Distribution UserDefinedData_t 
        │       └───Element DataArray_t I4 [   0 5000 5000]
        ├───TRI_3.0 Elements_t I4 [5 0]
        │   ├───ElementRange IndexRange_t I4 [5001 6200]
        │   ├───